# Lab 05 · Agent-to-agent communication with A2A

**Prepared by Mustafa Assaf** · Beginner level · 60–90 minute session

Start with the small worked example, read the expected output, then change one thing.
Use a CPU runtime and run the cells from top to bottom. No repository clone is needed.

**Practice mode:** model answers are explicitly scripted so you can inspect the code
without a key. They demonstrate control flow, not model quality. MCP, A2A and LangGraph
examples run their actual libraries locally even in practice mode; their setup cells
install pinned packages if needed.

**Optional live calls:** add `FREEINFERENCE_API_KEY` in Colab's Secrets panel, allow
this notebook access, and change `MODE` to `"live"`. The initial model is
`qwen3.6-35b` on FreeInference; `BASE_URL`, `MODEL` and `SECRET_NAME` can be changed
for another compatible provider. Keep the key out of cells and outputs.

### Shared setup
The next cell contains the reusable model client. Run it as supplied; the short teaching examples follow below.

In [ ]:
"""Small, provider-configurable teaching client. Standard library only."""
import getpass
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request

# Shared by every Client in this notebook/process, including parallel workflows.
_NETWORK_SLOTS = threading.BoundedSemaphore(2)


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None  # Never forward a credential to another host.


class Client:
    """Practice responses are explicit fixtures, not an LLM or model simulation."""
    def __init__(self, mode="practice", model="", family="", base_url="", api_key="",
                 max_calls=24, max_tokens=512, timeout=45, token_field="max_tokens", extra_body=None):
        if extra_body is not None and (not isinstance(extra_body, dict) or
                set(extra_body) & {"model", "messages", "max_tokens", "max_completion_tokens", "stream"}):
            raise ValueError("extra_body must be a dict and cannot override the request contract or budgets")
        if mode not in {"practice", "live"}:
            raise ValueError("mode must be practice or live")
        if token_field not in {"max_tokens", "max_completion_tokens"}:
            raise ValueError("Check which output-token field your provider supports")
        if max_calls < 1 or max_tokens < 1 or timeout <= 0:
            raise ValueError("Call, output, and timeout bounds must be positive")
        if mode == "live":
            url = urllib.parse.urlsplit(base_url)
            if url.scheme != "https" or not url.netloc or url.username or url.password or url.query or url.fragment:
                raise ValueError("Use an HTTPS API base URL, without credentials/query/fragment")
            if not model.strip() or not api_key.strip():
                raise ValueError("Live mode requires a model and secret key")
        self.mode, self.model, self.family = mode, model, family
        self.base_url, self._api_key = base_url.rstrip("/"), api_key
        self.max_calls, self.max_tokens, self.timeout = max_calls, max_tokens, timeout
        self.token_field, self.trace = token_field, []
        self.extra_body = dict(extra_body or {})
        self._freeinference = urllib.parse.urlsplit(self.base_url).hostname == "freeinference.org"
        self._attempts, self._lock = 0, threading.Lock()

    def ask(self, prompt, *, system="You are a concise teaching assistant.", practice=""):
        if not isinstance(prompt, str) or len(prompt) + len(system) > 28000:
            raise ValueError("Prompt must be text and total prompt length <= 28,000 characters")
        with self._lock:
            if self._attempts >= self.max_calls:
                raise RuntimeError("Call budget reached; inspect the trace before increasing it")
            self._attempts += 1
            call_id = self._attempts
        started = time.perf_counter()
        row = dict(call=call_id, mode=self.mode, model=self.model or "scripted-fixture",
                   family=self.family or ("practice" if self.mode == "practice" else None), input_tokens=None, output_tokens=None,
                   cost=None, status="ok")
        try:
            if self.mode == "practice":
                content = practice
            else:
                messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
                body = {"model": self.model, "messages": messages, self.token_field: self.max_tokens}
                body.update(self.extra_body)
                if self._freeinference:
                    body.update({"stream": False, "thinking": {"type": "disabled"}})
                req = urllib.request.Request(self.base_url + "/chat/completions",
                    data=json.dumps(body).encode(), method="POST",
                    headers={"Authorization": "Bearer " + self._api_key, "Content-Type": "application/json"})
                try:
                    with _NETWORK_SLOTS:
                        with urllib.request.build_opener(_NoRedirect).open(req, timeout=self.timeout) as response:
                            payload = json.load(response)
                except urllib.error.HTTPError as exc:
                    if exc.code == 429:
                        raise RuntimeError("Gateway HTTP 429: rate/quota limit. Check your quota; no retry or key rotation made.") from None
                    raise RuntimeError(f"Gateway HTTP {exc.code}; check provider access/configuration. No retry made.") from None
                except (urllib.error.URLError, TimeoutError, OSError, ValueError):
                    raise RuntimeError("Gateway/network/JSON error; inspect provider configuration. No retry made.") from None
                try:
                    choice = payload["choices"][0]
                    if choice.get("finish_reason") == "length":
                        raise RuntimeError("Output truncated; inspect token budget before retrying")
                    content = choice["message"]["content"]
                    if not isinstance(content, str):
                        raise ValueError("Missing text response")
                    usage = payload.get("usage") or {}
                    row["input_tokens"] = usage.get("prompt_tokens")
                    row["output_tokens"] = usage.get("completion_tokens")
                    if self._freeinference and "model is starting up" in content.lower():
                        raise RuntimeError("FreeInference model is starting up. Wait about 120 seconds, then retry manually; this attempt used the call budget.")
                except (KeyError, IndexError, TypeError, ValueError, AttributeError):
                    raise RuntimeError("Gateway response has no usable text; schema/capability check needed") from None
            return content
        except Exception:
            row["status"] = "error"
            raise
        finally:
            row["latency_s"] = round(time.perf_counter() - started, 4)
            with self._lock:
                self.trace.append(row)


def secret(name="FREEINFERENCE_API_KEY"):
    """Read a Colab secret, then fall back to a hidden local prompt. Never print it."""
    try:
        from google.colab import userdata
        key = userdata.get(name)
        if key:
            return key
    except Exception:
        pass
    return getpass.getpass(f"{name} (hidden): ")


def parse_object(text):
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError("Expected one JSON object")
    return value


def summarize_trace(clients):
    rows = [row for client in clients for row in client.trace]
    return dict(calls=len(rows), failed_calls=sum(r["status"] != "ok" for r in rows),
                input_tokens=sum(r["input_tokens"] for r in rows) if rows and all(isinstance(r["input_tokens"], int) for r in rows) else None,
                output_tokens=sum(r["output_tokens"] for r in rows) if rows and all(isinstance(r["output_tokens"], int) for r in rows) else None,
                summed_call_latency_s=round(sum(r["latency_s"] for r in rows), 4), cost=None)

In [ ]:
# Run all examples without a key first. Change only MODE for a live model run.
MODE = "practice"  # "live" uses the API key supplied through Colab Secrets
BASE_URL = "https://freeinference.org/v1"
MODEL = "qwen3.6-35b"
SECRET_NAME = "FREEINFERENCE_API_KEY"

client = Client(mode=MODE, base_url=BASE_URL, model=MODEL,
                api_key=secret(SECRET_NAME) if MODE == "live" else "",
                max_calls=24, max_tokens=512)
print("Mode:", MODE)

## What you will learn

Discover an agent's **Agent Card**, send one **A2A message**, and read its reply. Allow 60–90 minutes, including the exercise. The whole example delegates one tiny task: greet a student by name.

This uses the official `a2a-sdk==1.3.0` and the A2A **1.0 JSON-RPC** binding. An HTTP test transport runs the real SDK server routes inside the notebook, so discovery, JSON serialisation, and protocol handling still happen. It opens no public endpoint or listening port and needs no LLM key. The agent's greeting behavior is deterministic so we can focus on the protocol.

## 1. Install and import the SDK

Run the collapsed setup cell. The first Colab run needs internet access to install the pinned package and HTTP-server dependencies. The example's subsequent requests stay inside the notebook process.

In [ ]:
import importlib.metadata
import subprocess
import sys

try:
    sdk_version = importlib.metadata.version("a2a-sdk")
except importlib.metadata.PackageNotFoundError:
    sdk_version = None
if sdk_version != "1.3.0":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "a2a-sdk[http-server]==1.3.0",
    ])

import httpx
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.helpers import get_message_text, new_text_message
from a2a.server.agent_execution import AgentExecutor
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import (
    AgentCapabilities, AgentCard, AgentInterface, AgentSkill,
    Role, SendMessageRequest,
)
from starlette.applications import Starlette

## 2. Give the agent one behavior

`execute` receives the user's message through A2A and places a reply on the SDK event queue. The caller does not need to know how the greeting is implemented. For this immediate response we do not create a long-running task.

In [ ]:
class Greeter(AgentExecutor):
    async def execute(self, context, event_queue):
        name = get_message_text(context.message).strip()
        await event_queue.enqueue_event(new_text_message(f"Hello, {name}!"))

    async def cancel(self, context, event_queue):
        raise NotImplementedError("This immediate reply does not create tasks.")

## 3. Publish a card and protocol routes

The collapsed cell declares the agent's name, `greet` skill, text input/output, and A2A 1.0 JSON-RPC endpoint. The SDK serves its card at `/.well-known/agent-card.json` and handles protocol requests.

`agent.local` is a label used only by the in-process test transport; it is not an external website. The card advertises no streaming support because this example returns one response.

In [ ]:
agent_card = AgentCard(
    name="Greeter",
    description="Greets one student by name.",
    version="1.0.0",
    default_input_modes=["text/plain"],
    default_output_modes=["text/plain"],
    capabilities=AgentCapabilities(streaming=False),
    supported_interfaces=[AgentInterface(
        protocol_binding="JSONRPC", url="http://agent.local/",
        protocol_version="1.0",
    )],
    skills=[AgentSkill(
        id="greet", name="Greet", description="Greet a student by name.",
        tags=["greeting"],
    )],
)
handler = DefaultRequestHandler(
    agent_executor=Greeter(), task_store=InMemoryTaskStore(),
    agent_card=agent_card,
)
app = Starlette(routes=(
    create_agent_card_routes(agent_card)
    + create_jsonrpc_routes(handler, "/")
))
transport = httpx.ASGITransport(app=app)

## 4. Discover, delegate, and read the reply

The caller first retrieves the card. `create_client` uses that card to select the protocol interface. A message carries the name, and the response carries the greeting. The SDK generates message IDs; no model prompt is pretending to be a protocol message.

Expected output:

```text
Discovered: Greeter
Hello, Ada!
```

In [ ]:
async with httpx.AsyncClient(transport=transport, base_url="http://agent.local") as http:
    resolver = A2ACardResolver(httpx_client=http, base_url="http://agent.local")
    discovered = await resolver.get_agent_card()
    print("Discovered:", discovered.name)
    config = ClientConfig(streaming=False, httpx_client=http)
    remote = await create_client(discovered, client_config=config)
    message = new_text_message("Ada", role=Role.ROLE_USER)
    async for event in remote.send_message(SendMessageRequest(message=message)):
        reply = get_message_text(event.message)
        print(reply)
    await remote.close()

## 5. Check the boundary

These checks establish successful discovery and a correctly transported reply. They do not claim that a greeting is intelligent or that a discovered agent is trustworthy.

In [ ]:
assert discovered.name == "Greeter"
assert discovered.skills[0].id == "greet"
assert discovered.supported_interfaces[0].protocol_version == "1.0"
assert reply == "Hello, Ada!"
print("Discovery and reply checks passed.")

## MCP and A2A: what changed?

| MCP lab | A2A lab |
| --- | --- |
| Discover a tool and its argument schema. | Discover an agent and its declared skills/interfaces. |
| Call `lookup_note(topic)`. | Send a message requesting a greeting. |
| The host directs tool execution. | A separate agent handles the delegated request. |

Both are interoperability protocols. A2A can carry a direct message, as here, or a task whose state changes over time. Task progress, cancellation, streaming, authentication, and deployment are useful next steps, but they are outside this small example. The presence of an Agent Card alone does not establish identity, capability quality, or permission.

## Your turn

Change `Ada` to your name and update the expected reply. Then change the agent's greeting and rerun the app setup and request. Explain which code changes belong to the caller and which belong to the receiving agent.

**Exit ticket:** What information did the Agent Card give the caller before it sent the message?

Further reading: [official A2A Python SDK](https://github.com/a2aproject/a2a-python), [A2A agent discovery](https://a2a-protocol.org/latest/topics/agent-discovery/), and [A2A and MCP](https://a2a-protocol.org/latest/topics/a2a-and-mcp/). The lesson uses original greeting logic with the official SDK's server and client APIs.

### Finish
Explain the result in your own words. Make the small change suggested above and rerun its check. If you use a live model, save the actual result and trace; a different or failed response is something to investigate.